## PyTorch 矢量化加速原理

矢量化加速的核心是**用一条指令同时处理多个数据**，避免逐元素的 Python 循环。

### Python 循环慢在哪里

```python
# 慢：每步都要解释执行、动态类型检查、Python 对象开销
for i in range(n):
    c[i] = a[i] + b[i]

# 快：一次调用，底层 C/CUDA 执行
c = a + b
```

Python 循环每次迭代都要经过解释器、GIL、对象引用计数，单次开销约是 C 的 100 倍以上。

### CPU 上的 SIMD

现代 CPU 有 AVX-512 等指令集，一条指令可以同时对 16 个 float32 做加法：

```
普通:  a[0]+b[0], a[1]+b[1], ...     每次 1 个
SIMD:  [a[0]~a[15]] + [b[0]~b[15]]   每次 16 个
```

PyTorch 底层调用 MKL / OpenBLAS，这些库已针对 SIMD 高度优化。

### GPU 上的大规模并行

GPU 有数千个 CUDA Core，矩阵乘法 $XW$ 的每个输出元素可以**同时**由不同核心计算：

| | 方式 | 时间 |
|--|--|--|
| CPU | 逐行逐列串行 | $O(m \cdot n \cdot k)$ |
| GPU | 所有元素并行 | 理论接近 $O(1)$（受带宽限制） |

PyTorch 调用 cuBLAS / cuDNN，这些是 NVIDIA 专门为矩阵运算优化的库。

### 内存连续性（cache 友好）

PyTorch 张量默认行优先连续存储，矢量化操作顺序读取内存，命中 L1/L2 缓存；而 Python 循环跳跃访问对象会频繁 cache miss，带来额外延迟。

### 小结

| 层面 | 机制 | 加速来源 |
|------|------|----------|
| Python 层 | 用张量运算替代循环 | 绕开解释器开销 |
| CPU | SIMD 指令（AVX-512） | 一条指令处理 16 个 float32 |
| GPU | 数千 CUDA Core 并行 | 矩阵元素同时计算 |
| 内存 | 连续存储、顺序访问 | 提高 cache 命中率 |

> 矢量化 = 把"Python 逐元素循环"换成"底层库的批量指令"，在 CPU 上靠 SIMD，在 GPU 上靠数千核心并行，两者都绕开了 Python 解释器的开销。